In [1]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths
import numpy as np
import pandas as pd
from typing import Dict, Tuple, List
from collections import Counter

In [ ]:
df = pd.read_parquet('data/QA2_Bandpass_Data_Labeled_Filtered_Scored.parquet')
df['uid'] = df.index
df = df.reset_index(drop=True)

In [5]:
df.shape[0]

79184

In [7]:
df.columns

Index(['eb_uid', 'time', 'receiver_band', 'ref_antenna_name', 'antenna_name',
       'baseband_name', 'spw_name_ms', 'pol_id', 'frequency_array',
       'amplitude', 'phase', 'QA2 Flag(s)', 'uid', 'atmospheric_interference',
       'score_masked', 'score_unmasked', 'score_fixed', 'kernel_size',
       'win_masked_start', 'win_masked_end', 'win_unmasked_start',
       'win_unmasked_end', 'win_fixed_start', 'win_fixed_end',
       'overlap_unmasked_pct', 'overlap_fixed_pct', 'fixed_bins_native',
       'rank_score'],
      dtype='object')

In [4]:
def _to_np(x):
    a = np.asarray(x, dtype=float)
    return a[np.isfinite(a)]

In [37]:
def classify_row(amplitude, min_len=8, abs_std_thresh=1e-6,
                 edge_frac=0.05, k_min=8, edge_abs_diff=3.0, center_std_max=0.5):
    y = _to_np(amplitude)
    n = y.size
    if n < min_len:
        return "regular"

    mu = y.mean()
    if np.max(np.abs(y - mu)) < abs_std_thresh:
        return "low_var"

    k = max(int(edge_frac * n), k_min)
    k = min(k, n // 4)

    left_mean  = float(np.mean(y[:k]))
    right_mean = float(np.mean(y[-k:]))

    center = y[k:-k] if n >= 2 * k + 1 else y
    c_med  = float(np.median(center))
    c_std  = float(np.std(center))

    left_dev  = left_mean  - c_med
    right_dev = right_mean - c_med

    if (c_std <= center_std_max and
        abs(left_dev) > edge_abs_diff and abs(right_dev) > edge_abs_diff and
        np.sign(left_dev) == np.sign(right_dev)):
        return "edge_plateau"

    return "regular"


In [38]:
def _majority_with_tiebreak(s):
    priority = {"edge_plateau": 2, "low_var": 1, "regular": 0}
    vc = s.value_counts()
    best, bestc = None, -1
    for k, c in vc.items():
        if c > bestc or (c == bestc and priority.get(k, -1) > priority.get(best, -1)):
            best, bestc = k, c
    return best

In [39]:
dfC = df.copy()
dfC["row_type"] = dfC["amplitude"].apply(classify_row)

grp_cols = ["eb_uid", "antenna_name", "spw_name_ms", "pol_id"]

grp_majority = (dfC
    .groupby(grp_cols, dropna=False)["row_type"]
    .apply(_majority_with_tiebreak)
    .rename("grp_type")
    .reset_index())

grp_counts = (dfC
    .groupby(grp_cols + ["row_type"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index())

df_labeled = df.merge(grp_majority, on=grp_cols, how="left")

df_lowvar       = df_labeled[df_labeled["grp_type"] == "low_var"].copy()
df_edge_plateau = df_labeled[df_labeled["grp_type"] == "edge_plateau"].copy()
df_regular      = df_labeled[df_labeled["grp_type"] == "regular"].copy()

In [40]:
df_lowvar.shape[0]

1809

In [41]:
df_edge_plateau.shape[0]

5945

In [44]:
df_regular.drop(columns=['grp_type'], inplace=True)

In [ ]:
df_regular.to_parquet("data/QA2_Bandpass_Data_Labeled_Filtered.parquet", engine='pyarrow', compression="zstd")

In [ ]:
specs = [np.array(x, dtype=float)
            for x in df['amplitude_corr_tsys'].tolist()]
freqs = [np.array(x, dtype=float)
            for x in df['frequency_array'].tolist()]
keep = [not np.all(s == 0.0) for s in specs]
specs = [s for s,k in zip(specs, keep) if k]
freqs = [f for f,k in zip(freqs, keep) if k]

uid = df['uid'].values[keep]
ref = df['ref_antenna_name'].values[keep]
ant = df['antenna'].values[keep]
pol = df['polarization'].values[keep]

In [ ]:
length_groups: Dict[int, List[int]] = {}
for i, s in enumerate(specs):
        L = s.shape[0]
        length_groups.setdefault(L, []).append(i)

In [ ]:
result: Dict[int, Tuple[np.ndarray, ...]] = {}
for L, idxs in length_groups.items():
        specs_L = np.vstack([specs[i] for i in idxs])
        freqs_L = np.vstack([freqs[i] for i in idxs])
        uid_L   = uid[idxs]
        ref_L   = ref[idxs]
        ant_L   = ant[idxs]
        pol_L   = pol[idxs]
        result[L] = (specs_L, uid_L, ref_L, ant_L, pol_L, freqs_L)

In [ ]:
L = 1920
specs_L, uid_L, ref_L, ant_L, pol_L, freqs_L = result[L]

endpoints = [(f.min(), f.max()) for f in freqs_L]

cnt = Counter(endpoints)

for (fmin, fmax), freq in cnt.items():
    print(f"Range = ({fmin:.3f}, {fmax:.3f})  →  {freq} rows")


In [ ]:
fmin_target, fmax_target = cnt.most_common(1)[0][0]
mask = np.array([
    (f.min() == fmin_target and f.max() == fmax_target)
    for f in freqs_L
])

uids_to_keep = uid_L[mask]
df_filtered = df[df["uid"].isin(uids_to_keep)].reset_index(drop=True)

In [ ]:
df_filtered.info()

In [ ]:
def match_and_correct(
    freq_array: np.ndarray,
    trans_freqs: np.ndarray,
    trans_vals: np.ndarray
) -> np.ndarray:
    idxs = np.searchsorted(trans_freqs, freq_array)
    idxs[idxs == len(trans_freqs)] = len(trans_freqs) - 1
    left  = np.maximum(idxs - 1, 0)
    right = idxs
    dl = np.abs(freq_array - trans_freqs[left])
    dr = np.abs(trans_freqs[right] - freq_array)
    nearest = np.where(dl <= dr, left, right)
    mt = trans_vals[nearest]
    return mt

In [ ]:
df_filtered['frequency_array'] = df_filtered['frequency_array'].apply(lambda s: np.asarray(s, dtype=float))
df_filtered['frequency_array'] = df_filtered['frequency_array'].apply(lambda freqs: [f/1e9 for f in freqs])

In [ ]:
trans_df = pd.read_parquet('data/full_spectrum.gzip')
trans_freqs = trans_df['Frequency (GHz)'].values
trans_vals  = trans_df['Transmission (%)'].values

In [ ]:
results = df_filtered.apply(
    lambda row: match_and_correct(
        np.array(row['frequency_array'], dtype=float),
        trans_freqs,
        trans_vals
    ),
    axis=1
)

In [ ]:
df_filtered['transmission_array'] = results

In [ ]:
interference = []
for index in df_filtered.index:
    freqs = np.array(df_filtered.loc[index, 'frequency_array'], dtype=float)
    trans = np.array(df_filtered.loc[index, 'transmission_array'], dtype=float)

    troguhs, props = find_peaks(-trans, prominence=1)
    _, _, left_ips, right_ips = peak_widths(-trans, troguhs, rel_height=0.75)

    left_freqs  = np.interp(left_ips,  np.arange(len(freqs)), freqs)
    right_freqs = np.interp(right_ips, np.arange(len(freqs)), freqs)
    widths_freq = right_freqs - left_freqs

    trough_freqs  = freqs[troguhs]
    trough_ranges = []
    for i in range(len(trough_freqs)):
        trough_ranges.append((trough_freqs[i] - widths_freq[i] / 2, trough_freqs[i] + widths_freq[i] / 2))
    trough_ranges = np.array(trough_ranges)

    closest_idxs = []
    for troguhs_range in trough_ranges:
        start, end = troguhs_range[0], troguhs_range[1]
        closest_start_idx = int(np.abs(freqs - start).argmin())
        closest_end_idx = int(np.abs(freqs - end).argmin())
        closest_idxs.append((closest_start_idx, closest_end_idx))
    interference.append(closest_idxs)

In [ ]:
df_filtered['atmospheric_interference'] = interference

In [ ]:
df_filtered.info()

In [ ]:
for col in ["transmission_array"]:
    df_filtered[col] = df_filtered[col].apply(lambda arr: arr.tolist())

In [ ]:
df_filtered.to_csv('Data/bandpass_filtered_same_freq.csv',index=None)